### 第 1 步：建立資料夾並下載 VOC2007

In [29]:
import os
import urllib.request
import tarfile

# 建立資料夾
voc_dir = "./VOCdevkit"
os.makedirs(voc_dir, exist_ok=True)

# VOC2007 trainval
voc2007_trainval_url = "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar"
voc2007_trainval_tar = os.path.join(voc_dir, "VOCtrainval_06-Nov-2007.tar")

# 下載 VOC2007 trainval (約 446MB)
if not os.path.exists(voc2007_trainval_tar):
    print("🔽 Downloading VOC2007 trainval...")
    urllib.request.urlretrieve(voc2007_trainval_url, voc2007_trainval_tar)
    print("✅ Download complete.")

# 解壓縮 VOC2007 trainval
with tarfile.open(voc2007_trainval_tar) as tar:
    tar.extractall(path=voc_dir)
    print("✅ VOC2007 trainval extracted.")


🔽 Downloading VOC2007 trainval...


KeyboardInterrupt: 

### 第 2 步：載入 VOC 資料（使用 torchvision）

In [ ]:
from torchvision.datasets import VOCDetection
from torchvision.transforms import ToTensor

# 載入 VOC2007 的 trainval 資料
dataset = VOCDetection(
    root="./VOCdevkit",
    year="2007",
    image_set="trainval",  # 可改為 "train" 或 "val"
    download=False,
    transform=ToTensor()
)

print(f"✅ 資料集大小：{len(dataset)} 張影像")

# 顯示第一筆資料的圖片與標註
img, target = dataset[0]
print("圖片尺寸：", img.shape)
print("標註內容：", target["annotation"]["object"])


✅ 資料集大小：5011 張影像
圖片尺寸： torch.Size([3, 375, 500])
標註內容： [{'name': 'chair', 'pose': 'Rear', 'truncated': '0', 'difficult': '0', 'bndbox': {'xmin': '263', 'ymin': '211', 'xmax': '324', 'ymax': '339'}}, {'name': 'chair', 'pose': 'Unspecified', 'truncated': '0', 'difficult': '0', 'bndbox': {'xmin': '165', 'ymin': '264', 'xmax': '253', 'ymax': '372'}}, {'name': 'chair', 'pose': 'Unspecified', 'truncated': '1', 'difficult': '1', 'bndbox': {'xmin': '5', 'ymin': '244', 'xmax': '67', 'ymax': '374'}}, {'name': 'chair', 'pose': 'Unspecified', 'truncated': '0', 'difficult': '0', 'bndbox': {'xmin': '241', 'ymin': '194', 'xmax': '295', 'ymax': '299'}}, {'name': 'chair', 'pose': 'Unspecified', 'truncated': '1', 'difficult': '1', 'bndbox': {'xmin': '277', 'ymin': '186', 'xmax': '312', 'ymax': '220'}}]


### 必要套件安裝與載入

In [ ]:
# 安裝必要套件（若未安裝）
!pip install -q matplotlib opencv-python pandas scikit-learn torchvision tqdm

import os
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from torchvision.datasets import VOCDetection
from torchvision.transforms import ToTensor
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm


### 載入 VOC 資料集並挑選特定類別（e.g., person, car, dog

In [ ]:
SELECTED_CLASSES = ["person", "car", "dog"]
MAX_IMAGES_PER_CLASS = 30

voc_root = "./VOCdevkit"
dataset = VOCDetection(root=voc_root, year="2007", image_set="trainval", download=False)

# 建立每類別對應的圖像索引
class_to_indices = {cls: [] for cls in SELECTED_CLASSES}

for idx in range(len(dataset)):
    _, target = dataset[idx]
    objs = target["annotation"]["object"]
    objs = [objs] if isinstance(objs, dict) else objs

    labels = [obj["name"] for obj in objs]
    for cls in SELECTED_CLASSES:
        if cls in labels and len(class_to_indices[cls]) < MAX_IMAGES_PER_CLASS:
            class_to_indices[cls].append(idx)


### 定義自動配對 (query, support)，並擷取支援影像 (bounding box)

In [ ]:
def crop_support_from_target(image, target, target_class):
    objs = target["annotation"]["object"]
    objs = [objs] if isinstance(objs, dict) else objs

    for obj in objs:
        if obj["name"] == target_class:
            bbox = obj["bndbox"]
            xmin, ymin = int(bbox["xmin"]), int(bbox["ymin"])
            xmax, ymax = int(bbox["xmax"]), int(bbox["ymax"])
            return image.crop((xmin, ymin, xmax, ymax))
    return None


### 模擬通道特徵（不使用模型，先用圖像統計當通道特徵）

In [ ]:
def compute_channel_statistics(image: Image.Image):
    # 將 PIL image 轉 numpy array
    img_np = np.array(image.resize((224, 224)))  # 模擬輸入尺寸
    if len(img_np.shape) == 2:  # 灰階
        img_np = np.stack([img_np]*3, axis=-1)

    stats = {}
    for c in range(3):  # 模擬 RGB 通道
        channel = img_np[..., c]
        stats[f"mean_c{c}"] = np.mean(channel)
        stats[f"std_c{c}"] = np.std(channel)
        stats[f"var_c{c}"] = np.var(channel)
        stats[f"contrast_c{c}"] = np.max(channel) - np.min(channel)
    return stats


### 批次執行分析，儲存 CSV

In [ ]:
results = []

for cls in SELECTED_CLASSES:
    indices = class_to_indices[cls]
    for idx in tqdm(indices, desc=f"分析類別 {cls}"):
        query_img, query_target = dataset[idx]
        support_img = crop_support_from_target(query_img, query_target, cls)
        if support_img is None:
            continue

        stats = compute_channel_statistics(query_img)
        stats_support = compute_channel_statistics(support_img)

        stats.update({f"support_{k}": v for k, v in stats_support.items()})
        stats["class"] = cls
        stats["image_id"] = query_target["annotation"]["filename"]
        results.append(stats)

# 儲存成 CSV
df = pd.DataFrame(results)
df.to_csv("voc_channel_analysis.csv", index=False)
df.head()


分析類別 dog: 100%|██████████| 30/30 [00:00<00:00, 132.74it/s]


,mean_c0,std_c0,var_c0,contrast_c0,mean_c1,std_c1,var_c1,contrast_c1,mean_c2,std_c2,...,support_mean_c1,support_std_c1,support_var_c1,support_contrast_c1,support_mean_c2,support_std_c2,support_var_c2,support_contrast_c2,class,image_id
0,80.133590,49.494929,2449.748038,254,103.562879,46.887103,2198.400383,255,64.775690,56.496692,...,67.138413,54.291779,2947.597292,255,53.277503,45.499622,2070.215602,255,person,000009.jpg
1,136.663863,57.891876,3351.469282,255,136.649175,55.882669,3122.872717,255,109.587871,65.103327,...,114.323561,76.327008,5825.812181,255,99.204002,80.182173,6429.180800,255,person,000017.jpg
2,92.643535,58.261578,3394.411517,255,94.984435,55.519187,3082.380119,255,103.149135,54.497728,...,92.439473,48.366039,2339.273720,250,113.527463,47.950050,2299.207313,253,person,000021.jpg
3,137.002292,79.038395,6247.067816,255,121.321508,69.608601,4845.357331,255,109.892977,68.683229,...,115.131079,73.942879,5467.549364,255,117.839326,77.237492,5965.630234,255,person,000023.jpg
4,143.458945,62.967145,3964.861357,253,146.969328,63.135154,3986.047708,248,136.006776,74.782638,...,88.317303,64.406029,4148.136623,255,82.538146,61.054570,3727.660495,255,person,000030.jpg


### 通道分布分析與可視化

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ---------- 資料讀取與準備 ----------
df = pd.read_csv("voc_channel_analysis.csv")
output_dir = "output_charts"
os.makedirs(output_dir, exist_ok=True)

# 將欄位名稱改為 RGB
df.rename(columns={
    "mean_c0": "mean_R", "mean_c1": "mean_G", "mean_c2": "mean_B",
    "support_mean_c0": "support_mean_R", "support_mean_c1": "support_mean_G", "support_mean_c2": "support_mean_B"
}, inplace=True)

numeric_cols = df.select_dtypes(include=["number"]).columns
grouped = df.groupby("class")[numeric_cols].mean()

# ---------- 1. 各類別 RGB 通道平均值 ----------
plt.figure(figsize=(10, 6))
grouped[[col for col in grouped.columns if "mean_" in col]].plot(
    kind="bar", ax=plt.gca(), title="Mean RGB Value per Class", ylabel="Mean Pixel Value")
plt.tight_layout()
plt.savefig(f"{output_dir}/class_mean_rgb_barplot.png")
plt.close()

# ---------- 2. RGB 通道在各類別的分布差異（箱形圖） ----------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ch in enumerate(["R", "G", "B"]):
    df.boxplot(column=f"mean_{ch}", by="class", ax=axes[i])
    axes[i].set_title(f"Distribution of mean_{ch} by Class")
fig.suptitle("")
plt.tight_layout()
plt.savefig(f"{output_dir}/mean_rgb_boxplot_by_class.png")
plt.close()

# ---------- 3. 類別 vs 通道均值（熱度圖） ----------
heatmap_cols = [f"mean_{ch}" for ch in ["R", "G", "B"]] + [f"support_mean_{ch}" for ch in ["R", "G", "B"]]
plt.figure(figsize=(10, 6))
sns.heatmap(grouped[heatmap_cols], annot=True, cmap="YlGnBu")
plt.title("Class vs. RGB Channel Mean (Query & Support)")
plt.xlabel("Channel")
plt.ylabel("Class")
plt.tight_layout()
plt.savefig(f"{output_dir}/class_vs_channel_mean_heatmap.png")
plt.close()

# ---------- 4. 各類別內部的 RGB 通道變異（Intra-class Std Dev） ----------
class_std = df.groupby("class")[[f"mean_{ch}" for ch in ["R", "G", "B"]]].std()
plt.figure(figsize=(10, 6))
class_std.plot(kind="bar", ax=plt.gca(), rot=0)
plt.title("Intra-class Std Dev of RGB Channels")
plt.ylabel("Standard Deviation")
plt.xlabel("Class")
plt.legend(["Red (R)", "Green (G)", "Blue (B)"])
plt.tight_layout()
plt.savefig(f"{output_dir}/classwise_rgb_std_barplot.png")
plt.close()

# ---------- 5. 通道在不同類別之間的變異（Inter-class Std Dev） ----------
class_mean = df.groupby("class")[[f"mean_{ch}" for ch in ["R", "G", "B"]]].mean()
channel_variation_across_classes = class_mean.std()
plt.figure(figsize=(8, 5))
channel_variation_across_classes.plot(kind="bar", color=["red", "green", "blue"])
plt.title("Inter-class Variation of RGB Channel Means")
plt.ylabel("Std Dev Across Classes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{output_dir}/interclass_channel_variation_barplot.png")
plt.close()


In [ ]:
%cd ./Week4/

c:\Users\marti\Desktop\ntut-project\os2d\os2d\Week4


### 針對模型特徵圖計算通道統計資訊

In [ ]:
import torch
import numpy as np

def compute_feature_map_statistics(feature_map: torch.Tensor):
    """
    針對 ResNet50 的特徵圖輸出 (B, C, H, W) 計算每個通道的平均、標準差、變異數與對比度。
    """
    # 若為 batched tensor (1, C, H, W) → 移除 batch 維度
    if feature_map.dim() == 4:
        feature_map = feature_map.squeeze(0)  # 變成 (C, H, W)

    stats = {}
    for c in range(feature_map.shape[0]):  # C 通道
        channel_tensor = feature_map[c].detach().cpu().numpy()

        stats[f"mean_c{c}"] = np.mean(channel_tensor)
        stats[f"std_c{c}"] = np.std(channel_tensor)
        stats[f"var_c{c}"] = np.var(channel_tensor)
        stats[f"contrast_c{c}"] = np.max(channel_tensor) - np.min(channel_tensor)

    return stats


In [ ]:
%cd ..

c:\Users\marti\Desktop\ntut-project\os2d\os2d


In [ ]:
import torch
import torchvision.transforms as transforms
from os2d.modeling.model import build_os2d_from_config
from os2d.config import cfg
from os2d.utils import setup_logger

# 建立 logger
logger = setup_logger("OS2D")

# 使用 GPU（如果有）
cfg.is_cuda = torch.cuda.is_available()

# 載入預訓練模型（請確認此路徑存在該檔案）
cfg.init.model = "models/os2d_v2-train.pth"

# 初始化模型
net, box_coder, criterion, img_normalization, optimizer_state = build_os2d_from_config(cfg)

# 運算設備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 定義圖片前處理流程（與 OS2D 模型訓練設定一致）
transform_image = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=img_normalization["mean"], std=img_normalization["std"])
])


2025-03-25 14:50:06,654 OS2D INFO: Building the OS2D model
2025-03-25 14:50:07,055 OS2D INFO: Creating model on one GPU
2025-03-25 14:50:07,083 OS2D INFO: Reading model file models/os2d_v2-train.pth
2025-03-25 14:50:07,179 OS2D INFO: Loaded complete model from checkpoint
2025-03-25 14:50:07,179 OS2D INFO: Cannot find 'optimizer' in the checkpoint file. Initializing optimizer from scratch.
2025-03-25 14:50:07,182 OS2D INFO: OS2D has 139 blocks of 10169478 parameters (before freezing)
2025-03-25 14:50:07,183 OS2D INFO: OS2D has 139 blocks of 10169478 trainable parameters


c:\Users\marti\Desktop\ntut-project\os2d\os2d\os2d\modeling\model.py:306: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path)


In [30]:
%cd Week4/

c:\Users\marti\Desktop\ntut-project\os2d\os2d\Week4


In [32]:
from tqdm import tqdm
import pandas as pd

results = []

for cls in SELECTED_CLASSES:  # 你事先指定的類別清單，如 ["car", "person", "dog"]
    indices = class_to_indices[cls]  # 你事先建立好的 dict：class_name -> image indices
    for idx in tqdm(indices, desc=f"Analyzing class: {cls}"):
        query_img, query_target = dataset[idx]

        # 前處理成 Tensor
        query_img_th = transform_image(query_img).unsqueeze(0).to(device)
        support_img = crop_support_from_target(query_img, query_target, cls)
        if support_img is None:
            continue
        support_img_th = transform_image(support_img).unsqueeze(0).to(device)

        # 提取 feature map
        query_feat = net.net_feature_maps(query_img_th)
        support_feat = net.net_feature_maps(support_img_th)

        # 通道統計
        stats_query = compute_feature_map_statistics(query_feat)
        stats_support = compute_feature_map_statistics(support_feat)

        # 合併統計
        combined = stats_query
        combined.update({f"support_{k}": v for k, v in stats_support.items()})
        combined["class"] = cls
        combined["image_id"] = query_target["annotation"]["filename"]

        results.append(combined)

# 輸出 CSV
df = pd.DataFrame(results)
df.to_csv("./voc_featuremap_channel_analysis.csv", index=False)
print("✅ CSV 已輸出：voc_featuremap_channel_analysis.csv")


Analyzing class: person:   0%|          | 0/30 [00:00<?, ?it/s]

Analyzing class: dog: 100%|██████████| 30/30 [00:11<00:00,  2.54it/s]


✅ CSV 已輸出：voc_featuremap_channel_analysis.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ---------- 資料讀取與處理 ----------
df = pd.read_csv("./voc_featuremap_channel_analysis.csv")
output_dir = "output_top_channel_stats"
os.makedirs(output_dir, exist_ok=True)

# 選出所有通道均值與標準差欄位
mean_cols = [col for col in df.columns if col.startswith("support_mean_c")]
std_cols = [col for col in df.columns if col.startswith("support_std_c")]

# 計算各通道的平均值與標準差
mean_scores = df[mean_cols].mean().sort_values(ascending=False)
std_scores = df[std_cols].mean().sort_values(ascending=False)

# 取出前 1024 名
top1024_mean = mean_scores.head(1024)
top1024_std = std_scores.head(1024)

# ---------- 圖表 1：活躍度 Top 1024 ----------
plt.figure(figsize=(20, 6))
top1024_mean.plot(kind="bar", color="skyblue")
plt.title("Top 1024 Most Active Channels (by Mean Value)")
plt.ylabel("Average Activation")
plt.xticks([], [])  # 隱藏 X 軸文字，太多了
plt.tight_layout()
plt.savefig(f"{output_dir}/top1024_active_channels.png")
plt.close()

# ---------- 圖表 2：變異性 Top 1024 ----------
plt.figure(figsize=(20, 6))
top1024_std.plot(kind="bar", color="salmon")
plt.title("Top 1024 Most Variant Channels (by Std Dev)")
plt.ylabel("Average Standard Deviation")
plt.xticks([], [])  # 同樣隱藏
plt.tight_layout()
plt.savefig(f"{output_dir}/top1024_variable_channels.png")
plt.close()

top1024_mean.to_csv(f"{output_dir}/top1024_active_channels.csv")
top1024_std.to_csv(f"{output_dir}/top1024_variable_channels.csv")

# 取出前 500 名
top500_mean = mean_scores.head(500)
top500_std = std_scores.head(500)

# ---------- 圖表 1：活躍度 Top 500 ----------
plt.figure(figsize=(20, 6))
top500_mean.plot(kind="bar", color="skyblue")
plt.title("Top 500 Most Active Channels (by Mean Value)")
plt.ylabel("Average Activation")
plt.xticks([], [])  # 隱藏 X 軸文字，太多了
plt.tight_layout()
plt.savefig(f"{output_dir}/top500_active_channels.png")
plt.close()

# ---------- 圖表 2：變異性 Top 500 ----------
plt.figure(figsize=(20, 6))
top500_std.plot(kind="bar", color="salmon")
plt.title("Top 500 Most Variant Channels (by Std Dev)")
plt.ylabel("Average Standard Deviation")
plt.xticks([], [])  # 同樣隱藏
plt.tight_layout()
plt.savefig(f"{output_dir}/top500_variable_channels.png")
plt.close()

# 選出通道欄位
mean_cols = [col for col in df.columns if col.startswith("support_mean_c")]
std_cols = [col for col in df.columns if col.startswith("support_std_c")]

# 計算各通道的平均與標準差
mean_scores = df[mean_cols].mean().sort_values(ascending=False)
std_scores = df[std_cols].mean().sort_values(ascending=False)

# 提取前 500 通道
top500_mean = mean_scores.head(500)
top500_std = std_scores.head(500)

# ---------- 修正：找出交集通道 ID ----------
mean_ids = [col.replace("support_mean_c", "") for col in top500_mean.index]
std_ids = [col.replace("support_std_c", "") for col in top500_std.index]
common_ids = set(mean_ids).intersection(set(std_ids))

# ---------- 交集通道欄位 ----------
common_mean_cols = [f"support_mean_c{i}" for i in common_ids]
common_std_cols = [f"support_std_c{i}" for i in common_ids]

# ---------- 整理成 dataframe ----------
overlap_means = mean_scores[common_mean_cols].sort_values(ascending=False)
overlap_stds = std_scores[[f"support_std_c{c.split('_c')[1]}" for c in overlap_means.index]]

df_overlap = pd.DataFrame({
    'mean': overlap_means,
    'std': overlap_stds
})

# ---------- 圖表 ----------
fig, ax1 = plt.subplots(figsize=(20, 6))
ax1.plot(df_overlap['mean'].values, label="Mean Activation", color="mediumseagreen")
ax1.set_ylabel("Mean", color="mediumseagreen")
ax1.tick_params(axis='y', labelcolor="mediumseagreen")
ax1.set_title("Channels in Both Top 500 (Mean & Std Dev)")

ax2 = ax1.twinx()
ax2.plot(df_overlap['std'].values, label="Std Deviation", color="orange")
ax2.set_ylabel("Std Dev", color="orange")
ax2.tick_params(axis='y', labelcolor="orange")

fig.tight_layout()
plt.savefig(f"{output_dir}/top500_overlap_channels_dual_axis.png")
plt.close()

# ---------- 儲存資料 ----------
df_overlap.to_csv(f"{output_dir}/top500_overlap_channels.csv")
pd.Series(sorted(common_ids)).to_csv(f"{output_dir}/top500_overlap_channel_names.csv", index=False)


In [40]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# ---------- 讀取資料 ----------
df = pd.read_csv("./voc_featuremap_channel_analysis.csv")
output_dir = "output_top_channel_stats"
os.makedirs(output_dir, exist_ok=True)

# ---------- 找出所有通道 mean/std 欄位 ----------
mean_cols = [col for col in df.columns if col.startswith("support_mean_c")]
std_cols = [col for col in df.columns if col.startswith("support_std_c")]

# ---------- 計算平均值 ----------
mean_scores = df[mean_cols].mean()
std_scores = df[std_cols].mean()

# ---------- 對齊通道 ID ----------
mean_ids = [c.replace("support_mean_c", "") for c in mean_scores.index]
std_ids = [c.replace("support_std_c", "") for c in std_scores.index]
common_ids = sorted(set(mean_ids).intersection(set(std_ids)))

# ---------- 組合成分數表 ----------
combined_scores = pd.DataFrame({
    "channel": common_ids,
    "mean": [mean_scores[f"support_mean_c{i}"] for i in common_ids],
    "std": [std_scores[f"support_std_c{i}"] for i in common_ids]
})
combined_scores["score"] = combined_scores["mean"] + combined_scores["std"]

# ---------- 挑選建議剪枝通道（低 mean + std） ----------
bottom500 = combined_scores.sort_values(by="score", ascending=True).head(500)

# ---------- 畫圖 ----------
plt.figure(figsize=(24, 6))
plt.bar(range(len(bottom500)), bottom500["score"], color="gray")
plt.title("Recommended Bottom 500 Channels to Prune (Low Mean + Low Std)")
plt.ylabel("Mean + Std Score")
plt.xlabel("Channel Index")
plt.tight_layout()
plt.savefig(f"{output_dir}/bottom500_channels_to_prune.png")
plt.close()

# ---------- 儲存建議剪枝通道資訊 ----------
bottom500.to_csv(f"{output_dir}/bottom500_channels_to_prune.csv", index=False)
